# 3.0b — Tenseurs : strides, vues et mémoire

**Navigation** : [<< 3.0 — Théorie de l'information](3.0-Theorie-Information.ipynb) · [3.1 — Rétropropagation >>](3.1-Retropropagation.ipynb) · [Feuille de route de la série](README.md)

Toute opération « sans copie » que l'on nous vend — transposée, découpage, diffusion (broadcast) — n'est pas de la magie : c'est une **arithmétique d'adresses**. Un tableau ne stocke ni lignes ni colonnes : il stocke **un buffer plat** plus une recette pour s'y promener — la forme (*shape*) et les **strides** (pas, en octets, pour avancer le long de chaque dimension).

**Comment lire ce carnet.** Chaque section suit le même mouvement : une construction mesurée, puis la lecture de sa sortie, puis ce que la mesure interdit d'affirmer. Les quatre exercices s'intercalent aux endroits où la main doit prendre le relais de l'œil — ils sont faisables sans quitter le carnet, et leurs cellules s'exécutent sans erreur même non complétés. Les valeurs d'horloge et d'octets citées dans la prose sont celles imprimées par les cellules de ce run : elles varient d'une machine à l'autre, et c'est voulu — le carnet enseigne des *rapports* mesurés (un facteur, un ratio de stockage), pas des constantes.

Ce carnet ouvre cette mécanique, dans la discipline de la série : **d'abord NumPy pur** (le buffer et les strides y sont presque nus), **puis parité avec PyTorch** — le même contrat d'adresse, plus deux témoins propres à torch (`view` qui exige la contiguïté, le gradient qui traverse les vues). Chaque construction par `as_strided` est vérifiée contre un calcul d'indice fait à la main ; chaque affirmation de coût (octets, temps) est **mesurée dans ce carnet**, pas annoncée.

In [1]:
import time

import numpy as np
import torch

print("numpy", np.__version__, "| torch", torch.__version__)

numpy 2.4.2 | torch 2.14.0+cu126


## 1. L'identité d'un tableau : un buffer, une forme, des pas

Créons un tableau banal et demandons-lui ses papiers : où sont ses données (le buffer), à quel élément il commence (l'*offset*), et quelle recette il suit pour se déplacer (les *strides*). La méthode `.tobytes()` du buffer révèle la vérité physique : une seule bande plate.

Trois interrogations distinctes sont ici emmêlées, et le reste du carnet les démêle une à une. **Où sont les valeurs ?** Dans le buffer plat, et nulle part ailleurs — la sortie montre les douze entiers rangés à la queue. **Comment lire ce plat comme une grille ?** Par la recette (forme, strides) : c'est une *convention de lecture*, pas une structure. **Qu'est-ce que modifier `a` modifie-t-il ?** Le buffer — donc tout autre tableau qui lit le même buffer (la section 6 en fait un piège, l'autograd de torch en fait un atout).

Un réflexe à installer dès maintenant : devant un tableau inconnu, demander `shape`, `strides` et `data_ptr()` avant toute hypothèse sur ce qu'il coûte. Les trois réponses disent si les opérations suivantes seront des réécritures de contrat (gratuites) ou des copies (payées en octets et en temps).

In [2]:
a = np.arange(12, dtype=np.int64).reshape(3, 4)
print("a =\n", a)
print("forme       :", a.shape)
print("strides     :", a.strides, "(octets)")
print("offset      :", 0, "(premier element du buffer)")
buf = np.frombuffer(a.tobytes(), dtype=np.int64)
print("buffer plat :", buf)
print("taille buffer :", buf.nbytes, "octets =", buf.size, "elements x", buf.itemsize)

a =
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
forme       : (3, 4)
strides     : (32, 8) (octets)
offset      : 0 (premier element du buffer)
buffer plat : [ 0  1  2  3  4  5  6  7  8  9 10 11]
taille buffer : 96 octets = 12 elements x 8


Le tableau 3×4 n'occupe aucune structure 2D en mémoire : c'est la bande plate `0..11`, et le stride `(32, 8)` dit : *pour avancer d'une ligne, saute 32 octets (4 éléments) ; d'une colonne, 8 octets (1 élément)*. L'adresse de l'élément $(i, j)$ est donc une formule :

$$\text{offset} + i \cdot 32 + j \cdot 8 \quad \text{octets}$$

La lecture à faire de cette sortie est double. D'abord **l'unité** : les strides sont comptés en *octets*, pas en éléments — 32 octets parce qu'une ligne de 4 entiers de 8 octets en fait 32, et 8 octets pour un pas de colonne. Ensuite **l'ordre** : le premier stride (32) est plus grand que le second (8), signe d'un rangement *par lignes* (ordre C, pour *row-major*). Un tableau rangé par colonnes (ordre F, comme Fortran) échangerait les deux rôles. La formule ci-dessus n'est donc pas une propriété de NumPy mais un choix de convention — et tout ce qui suit (contiguïté, `view`, im2col) se joue sur ce choix.

NumPy appelle `as_strided` la construction inverse : donner un buffer, une forme et des pas, et obtenir un tableau qui **lit ce buffer** selon cette recette. Construisons une matrice 2×3 dans le même buffer, avec des pas de notre choix, et vérifions la formule à la main, élément par élément.

In [3]:
from numpy.lib.stride_tricks import as_strided

buf = np.arange(12, dtype=np.int64)
v = as_strided(buf, shape=(2, 3), strides=(16, 8))  # pas donnes en OCTETS
print("vue v =\n", v)

# Verification a la main : la formule d'adresse contre le contenu reel
ok = True
for i in range(2):
    for j in range(3):
        attendu = buf.view(np.int64)[(i * 16 + j * 8) // 8]  # octets -> indice element
        if v[i, j] != attendu:
            ok = False
print("formule d'adresse verifiee sur les 6 elements :", ok)

vue v =
 [[0 1 2]
 [2 3 4]]
formule d'adresse verifiee sur les 6 elements : True


La vue 2×3 n'a **rien copié** : ses six valeurs sont lues dans le buffer d'origine aux adresses dictées par nos pas. La vérification `True` porte sur les **six éléments** : pour chacun, l'adresse calculée par la formule (offset + pas) a été confrontée à l'élément que la vue affiche — la recette et l'affichage coïncident partout, pas seulement sur un coin.

Ce que `as_strided` apporte ici, et ce qu'il ne vérifie **pas** : la fonction ne bornifie rien. Nos pas `(2, 1)` en éléments restaient dans le buffer ; des pas plus larges que la forme, ou une forme plus grande que ce que le buffer contient, fabriqueraient un tableau qui **lit hors du buffer** — valeurs parasites ou crash, sans avertissement. C'est l'outil chirurgical du carnet : on ne l'emploie jamais sans la vérification à la main qui l'accompagne (exercice 1 ci-dessous).

C'est toute la mécanique — le reste du carnet l'exploite.

### Exercice 1 — l'indice linéaire, à la main

**Contexte** : avant tout geste de `as_strided`, savoir prédire ce que la recette lit.
**Objectif** : écrire `indice_lineaire(strides, i, j)` qui renvoie l'indice (en **éléments**, pas en octets) que l'adresse $(i, j)$ désigne dans le buffer.

```python
# Indice : strides est en octets — diviser par itemsize (8 pour int64).
# Etape 1 : convertir les deux strides en elements.
# Etape 2 : composer offset_element = i * s0_elem + j * s1_elem.
```

In [4]:
def indice_lineaire(strides, i, j, itemsize=8):
    '''Renvoie l'indice du buffer (en elements) lu par l'adresse (i, j).

    strides : couple d'entiers, en octets.
    '''
    # TODO etudiant
    return None  # TODO etudiant


# Auto-verification (a décommenter une fois completee) :
# assert indice_lineaire((16, 8), 1, 2) == 4   # element d'indice 4 du buffer
print("Exercice a completer")

Exercice a completer


## 2. Les vues gratuites : transposée, sous-blocs, dilatation, diffusion

La transposée `$a.T$` coûte-t-elle une copie ? Mesurons : si les deux tableaux pointent vers la même mémoire, l'opération n'a fait qu'**échanger les pas**.

« Gratuit » veut dire ici : *aucun octet de données n'a bougé*. Ce n'est pas « rapide » dans l'absolu — une vue reste assujettie à l'ordre de lecture de son buffer, et la section 3 mesurera précisément quand cet ordre se paie. Mais la construction elle-même ne touche qu'un petit objet d'en-tête (forme, pas, offset) : pour un tableau d'un million d'éléments comme pour douze, le coût de la transposée est le même — celui d'une fiche, pas celui d'une copie. Une conséquence pratique de cette économie, à garder en tête pour la suite : les bibliothèques que le carnet mesure (`sliding_window_view`, `broadcast_to`, l'`expand` de torch) sont toutes des **fabriques de contrats** au sens strict — elles rendent un objet dont les trois champs (offset, forme, pas) ont été réécrits et dont le buffer n'a pas bougé. C'est pourquoi leur coût de construction ne croît pas avec la taille du tableau, et pourquoi les mesurer « à vide » (construire puis jeter) ne renseigne sur rien : tout leur effet se joue dans les lectures et écritures qu'elles orchestrent ensuite.

La série de mesures qui suit transpose, découpe et dilate sans jamais allouer : chacune devrait afficher `memoire partagee : True`, et c'est le critère qui distingue une vue d'une copie dans tout ce carnet.

In [5]:
at = a.T
print("a.strides  :", a.strides)
print("a.T.strides:", at.strides, "(echanges, pas copies)")
print("memoire partagee :", np.shares_memory(a, at))
print("meme buffer, meme adresse de depart :",
      a.__array_interface__['data'][0] == at.__array_interface__['data'][0])

a.strides  : (32, 8)
a.T.strides: (8, 32) (echanges, pas copies)
memoire partagee : True
meme buffer, meme adresse de depart : True


Même adresse, pas échangés : la transposée est une **vue**. Le sous-bloc et la dilatation obéissent à la même économie — un slicing ne fait qu'écrire de nouveaux strides (et un nouvel offset) :

La sortie du bloc se lit comme une phrase, mot à mot. `strides (32, 16)` : le pas de **ligne** reste 32 (les lignes du bloc sont encore des lignes de `a`), le pas de **colonne** double à 16 (on ne garde qu'une colonne sur deux). `offset : 32 octets` : la vue commence un entier-cible plus loin — 32 octets, soit exactement la ligne 0 sautée par le `1:` du slicing. Chaque morceau de la syntaxe de slicing se retrouve, un à un, dans le triplet (offset, strides) : *traduction*, pas transformation. C'est la lecture à faire mentalement devant n'importe quel découpage : demander le nouveau contrat d'adresse avant de supposer quoi que ce soit sur le résultat — partagé, contigu, accessible en écriture.

In [6]:
bloc = a[1:, ::2]          # lignes 1.., colonnes une sur deux
print("bloc =\n", bloc)
print("strides du bloc :", bloc.strides, "| offset :", a[1:, ::2].__array_interface__['data'][0] - a.__array_interface__['data'][0], "octets")
print("memoire partagee avec a :", np.shares_memory(a, bloc))

bloc =
 [[ 4  6]
 [ 8 10]]
strides du bloc : (32, 16) | offset : 32 octets
memoire partagee avec a : True


La sortie du bloc précédent donne la recette complète : le slicing `[1:, ::2]` a doublé le pas de colonne (8 → 16 octets) en gardant le pas de ligne. La dilatation sur les *deux* axes est la généralisation immédiate — et le moment de vérifier que la syntaxe et le contrat d'adresse disent deux fois la même chose.

### Exercice 2 — la dilatation, sans slicing

**Contexte** : `a[::2, ::2]` dilate par sucre syntaxique ; la même vue s'écrit à la main.
**Objectif** : construire par `as_strided` la vue qui ne garde que les lignes et colonnes **paires** de `a` (3×4 → 2×2), et vérifier l'égalité avec `a[::2, ::2]`.

```python
# Indice : les pas d'origine sont (32, 8). Doubler chaque pas saute un element sur deux.
# Etape 1 : calculer la forme resultante (2, 2).
# Etape 2 : appeler as_strided(a, shape=..., strides=(64, 16)).
# Etape 3 : np.array_equal(vue, a[::2, ::2]) doit rendre True.
```

In [7]:
def dilater_par_strides(arr):
    '''Renvoie la vue (lignes paires, colonnes paires) construite par as_strided.'''
    # TODO etudiant
    return None  # TODO etudiant


# Auto-verification (a decommenter une fois completee) :
# v = dilater_par_strides(a)
# assert np.array_equal(v, a[::2, ::2]) and np.shares_memory(v, a)
print("Exercice a completer")

Exercice a completer


La **diffusion** (broadcast) est le sommet de cette économie : `np.broadcast_to` fabrique un tableau *logiquement* plus grand dont un stride vaut **zéro** — répéter sans stocker. À l'inverse, `np.repeat` matérialise chaque copie. La différence se lit dans les octets :

Le stride nul `(0, 8)` est la signature : avancer le long de l'axe diffusé ne coûte *aucun* octet de déplacement — toutes les « lignes » du tableau logique sont la même adresse, relue. Les octets réels mesurés rendent le contraste net : la vue n'occupe que la ligne stockée (32 octets), le `repeat` occupe le tableau entier (96). Le contenu est identique, la mémoire ne l'est pas — et cette asymétrie est un choix d'ingénierie quotidien : diffuser pour calculer (gratuit), matérialiser pour stocker (payé). Dans un entraînement, la même alternative se présente à chaque biais ajouté à un batch : le laisser en stride nul jusqu'à la dernière addition, ou le fondre dans les poids.

In [8]:
ligne = np.arange(4, dtype=np.int64)                # 4 elements
diffuse = np.broadcast_to(ligne, (3, 4))            # logiquement 3x4
repete = np.repeat(ligne.reshape(1, 4), 3, axis=0)  # reellement 3x4

print("broadcast : forme", diffuse.shape, "| strides", diffuse.strides, "(stride nul !)")
print("  octets reels sous la vue :", ligne.nbytes)
print("repeat    : forme", repete.shape, "| strides", repete.strides)
print("  octets reels            :", repete.nbytes)
print("contenu identique :", np.array_equal(diffuse, repete))

broadcast : forme (3, 4) | strides (0, 8) (stride nul !)
  octets reels sous la vue : 32
repeat    : forme (3, 4) | strides (32, 8)
  octets reels            : 96
contenu identique : True


Même contenu, **trois fois moins d'octets** pour la version diffusée : la ligne 0 du tableau diffusé et sa ligne 2 sont *la même mémoire*, relue. C'est gratuit tant qu'on ne fait que lire — et c'est pourquoi écrire dans un tableau diffusé est interdit (laquelle des trois lignes modifierait-on ?).

L'interdit n'est pas une coquetterie d'API : une écriture dans une vue à stride nul devrait écrire *trois fois* la même adresse pour trois indices logiques distincts — l'opération n'a pas de sens cohérent, et NumPy la refuse plutôt que de choisir. Retenir la règle générale : **une vue en lecture seule dès qu'un stride est nul**, une copie explicite (`np.ascontiguousarray`, `.contiguous()` en torch) si l'écriture est nécessaire.

## 3. Contiguïté : le prix de l'ordre séquentiel

Un tableau est **contigu** (ordre C) quand ses strides sont exactement ceux qu'exigerait un rangement séquentiel : parcourir `flat[k]` et `a[i, j]` dans le même ordre. La transposée, le sous-bloc, le stride nul brisent ce contrat. NumPy le dit dans `.flags`, et le coût se mesure :

La sortie drapeau les quatre cas : seul `a`, jamais découpé, reste contigu. Trois des quatre constructions « gratuites » de la section 2 le perdent — le prix caché de la gratuité en octets se paie ici, en ordre d'accès, et la mesure qui suit le quantifie.

In [9]:
print("a       contigu :", a.flags['C_CONTIGUOUS'], "| strides", a.strides)
print("a.T     contigu :", at.flags['C_CONTIGUOUS'], "| strides", at.strides)
print("bloc    contigu :", bloc.flags['C_CONTIGUOUS'], "| strides", bloc.strides)
print("diffuse contigu :", diffuse.flags['C_CONTIGUOUS'], "| strides", diffuse.strides)

a       contigu : True | strides (32, 8)
a.T     contigu : False | strides (8, 32)
bloc    contigu : False | strides (32, 16)
diffuse contigu : False | strides (0, 8)


Les quatre drapeaux imprimés ci-dessus forment le corpus de test idéal : un contigu (`a`) et trois non-contigus, chacun pour une raison différente (pas échangés, pas dilaté, pas nul). La fonction demandée ci-dessous doit rendre ces quatre verdicts *sans jamais construire les tableaux* — à partir de la seule recette (forme, pas), comme le ferait un diagnostic en amont d'un appel de bibliothèque.

### Exercice 3 — prédire la contiguïté depuis la recette

**Contexte** : les kernels compilés (BLAS, CUDA) exigent souvent des blocs contigus ; savoir le lire AVANT l'erreur.
**Objectif** : écrire `est_contigue(shape, strides, itemsize)` qui rend `True` ssi les strides correspondent au rangement séquentiel — sans tableau, juste la recette.

```python
# Indice : pour une forme (n0, n1), le rangement seq demande stride1 = itemsize
# et stride0 = n1 * itemsize. Generaliser : le stride de l'axe k est le produit
# des tailles des axes APRES k, fois itemsize.
# Etape 1 : calculer les strides attendus par cumul a partir de la droite.
# Etape 2 : comparer element par element aux strides donnes.
```

In [10]:
def est_contigue(shape, strides, itemsize=8):
    '''Rend True ssi (shape, strides) decrit un rangement seqentiel (ordre C).
    '''
    # TODO etudiant
    return None  # TODO etudiant


# Auto-verification (a decommenter une fois completee) :
# assert est_contigue((3, 4), (32, 8)) is True
# assert est_contigue((3, 4), (8, 32)) is False
# assert est_contigue((3, 4), (32, 0)) is False
print("Exercice a completer")

Exercice a completer


Pourquoi cet ordre importe : le matériel ne lit pas des éléments, il lit des **lignes de cache** de 64 octets — huit `float64` chargés d'un bloc, utiles ou non. Le contraste honnête n'est pas « la transposée est plus lente » (l'itérateur de NumPy réordonne si bien les boucles qu'une transposée peut paraître gratuite à l'horloge), c'est le **débit par élément utile** : sommons le buffer entier, puis un élément sur huit — un seul élément utile par ligne de 64 octets — avec un buffer plus grand que le cache, pour que la mesure ne soit pas servie par de la mémoire chaude :

Deux choix de protocole méritent d'être lus dans le code avant la sortie. Le buffer fait 128 Mo — bien plus que le cache L3 : chaque ligne de cache servie est une vraie visite en mémoire vive, pas une relecture tiède. Et la mesure retient le **meilleur** temps sur cinq essais : elle borne le comportement du matériel (trafic réseau, ordonnanceur), pas la chance. La vue `gros[:, ::8]` ne stocke rien de plus que `gros` — la somme parcourt 2 M d'éléments *utiles*, mais chacun réclame sa ligne de 64 octets entière.

In [11]:
gros = np.arange(16_000_000, dtype=np.float64).reshape(4000, 4000)  # 128 Mo, > cache
sub = gros[:, ::8]  # vue : 1 element utile par ligne de 64 octets

def meilleur_temps(f, n=5):
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t0)
    return min(ts)

t_plein = meilleur_temps(lambda: gros.sum())
t_sub = meilleur_temps(lambda: sub.sum())
debit_plein = gros.size / t_plein / 1e6
debit_sub = sub.size / t_sub / 1e6
print(f"somme pleine : {gros.size / 1e6:.1f} M elem en {t_plein * 1000:5.1f} ms -> {debit_plein:6.0f} M elem/s")
print(f"somme 1/8    : {sub.size / 1e6:.1f} M elem en {t_sub * 1000:5.1f} ms -> {debit_sub:6.0f} M elem/s")
print(f"debit par element utile degrade x{debit_plein / debit_sub:.1f} (mesure de CE run)")

somme pleine : 16.0 M elem en  16.8 ms ->    950 M elem/s
somme 1/8    : 2.0 M elem en   8.2 ms ->    243 M elem/s
debit par element utile degrade x3.9 (mesure de CE run)


L'horloge absolue n'est pas huit fois pire — mais chaque ligne de 64 octets n'y livre qu'un élément utile, et le **débit par élément utile** mesuré ci-dessus s'effondre d'un facteur mesuré, pas annoncé. C'est le vrai coût du désordre : la mémoire paie en trafic ce que l'arithmétique ne paie pas. C'est la raison d'être de `.contiguous()` (fabriquer la copie bien rangée) et, dans le monde CUDA, l'obsession des kernels pour les tuiles contiguës.

Pourquoi la dégradation mesurée (facteur ~4 sur ce run, le chiffre exact est imprimé par la cellule) reste loin du pire cas théorique (×8) : le préchargeur matériel lit les lignes suivantes tant que le bus est libre, et la boucle de sommation elle-même est simple pour l'itérateur. Entre les deux se loge toute l'ingénierie des accès mémoire — le facteur dépend de la taille du buffer, de la largeur du pas, de la machine ; c'est précisément pourquoi le carnet mesure au lieu d'affirmer. La leçon à emporter n'est pas un chiffre mais une posture : **devant une vue découpée qui traverse un grand tableau, demander le débit utile avant de blâmer l'algorithme**.

## 4. Parité PyTorch : le même contrat, deux témoins propres à torch

PyTorch applique le même contrat d'adresse — vérifions la parité des constructions, puis les deux comportements que NumPy n'a pas : `view` qui **exige** la contiguïté, et le **gradient** qui traverse les vues.

La parité mesurée se lit sur une seule unité de comptage : torch compte ses strides **en éléments** (`(4, 1)`), NumPy en **octets** (`(32, 8)`) — même recette, échelles différentes, et l'erreur classique qui consiste à comparer les deux tuples bruts se voit ici désamorcée par la double impression. Le `data_ptr` identique joue pour torch le rôle que jouait `shares_memory` chez NumPy : la transposée n'a pas bougé un octet.

In [12]:
ta = torch.arange(12, dtype=torch.int64).reshape(3, 4)
tv = torch.as_strided(ta, (2, 3), (2, 1))  # strides EN ELEMENTS chez torch
nv = as_strided(np.arange(12, dtype=np.int64), (2, 3), (16, 8))  # en OCTETS chez numpy

print("torch.as_strided (2,3),(2,1) =\n", tv.numpy())
print("numpy.as_strided (2,3),(16,8) =\n", nv)
print("parite des constructions :", np.array_equal(tv.numpy(), nv))
print("torch strides :", ta.stride(), "(en elements) | numpy strides :", np.arange(12).reshape(3, 4).strides, "(en octets)")
print("torch transpose :", ta.t().stride(), "| memoire : data_ptr identique =", ta.data_ptr() == ta.t().data_ptr())

torch.as_strided (2,3),(2,1) =
 [[0 1 2]
 [2 3 4]]
numpy.as_strided (2,3),(16,8) =
 [[0 1 2]
 [2 3 4]]
parite des constructions : True
torch strides : (4, 1) (en elements) | numpy strides : (32, 8) (en octets)
torch transpose : (1, 4) | memoire : data_ptr identique = True


**Témoin n° 1 — `view` exige la contiguïté.** Là où NumPy `reshape` choisit silencieusement vue-ou-copie, torch sépare : `view` ne fait **jamais** de copie — donc échoue bruyamment sur un tenseur non contigu. L'erreur est le comportement enseigné, pas un accident :

Le message d'erreur mérite une lecture attentive, car il nomme la cause exacte : *au moins une dimension s'étend sur deux sous-espaces contigus* — la transposée `(4, 3)` aux pas `(1, 4)` ne peut pas être lue comme une réinterprétation plate du stockage, aucune forme purement « aplatie » ne respecte ces pas. Et la sortie complète la leçon : `reshape(2, 6)` **réussit** lui, en fabriquant la copie que `view` refuse — `copie sous-jacente : True`. Deux méthodes, deux contrats : `view` = réécriture du contrat d'adresse seulement ; `reshape` = vue si possible, copie sinon. Choisir l'une ou l'autre, c'est choisir si l'on veut être averti du coût ou l'absorber en silence.

In [13]:
tt = ta.t()  # vue non contigue : forme (4, 3), strides (1, 4)
try:
    tt.view(2, 6)
except RuntimeError as e:
    print("view sur non contigu -> RuntimeError :")
    print(" ", e)
# reshape, lui, est transparent : vue si possible, copie sinon
print("reshape(2,6) reussit :", tt.reshape(2, 6).shape,
      "| copie sous-jacente :", tt.reshape(2, 6).untyped_storage().data_ptr() != ta.untyped_storage().data_ptr())

view sur non contigu -> RuntimeError :
  view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
reshape(2,6) reussit : torch.Size([2, 6]) | copie sous-jacente : True


**Témoin n° 2 — le coût mesuré de `.contiguous()`.** La copie bien rangée a un prix en octets ; l'opération `expand` (le broadcast de torch, stride nul) montre l'économie symétrique :

Les trois lignes de la sortie racontent le même aller-retour que chez NumPy, en strides comptés en éléments : `expand` fabrique le stride nul `(0, 1)` et ne stocke que la ligne source (32 octets), `contiguous()` déplie tout (96 octets). Le couple n'a de sens que lu ensemble : `expand` est l'économie maximale, `.contiguous()` en est la monnaie — on paie la copie exactement quand un consommateur en aval (un kernel CUDA, une sérialisation) exige l'ordre séquentiel. Entre les deux, l'allocateur de torch ne bouge pas : ce sont deux contrats de lecture sur le même stockage.

In [14]:
tl = torch.arange(4, dtype=torch.int64)
td = tl.expand(3, 4)                 # stride nul, 4 octets x 8 = 32 octets reels
print("expand(3,4) : strides", td.stride(), "| octets reels :", tl.element_size() * tl.numel())
tc = td.contiguous()                  # materialise les 3 lignes
print("contiguous() : strides", tc.stride(), "| octets reels :", tc.element_size() * tc.numel())
print("contenu identique :", torch.equal(td, tc))

expand(3,4) : strides (0, 1) | octets reels : 32
contiguous() : strides (4, 1) | octets reels : 96
contenu identique : True


## 5. Fenêtres glissantes et im2col : la convolution cachée dans les strides

Extraire toutes les fenêtres de largeur `w` d'une séquence semble exiger une boucle et des copies. C'est un pur problème de strides : `w` adresses qui se chevauchent. NumPy fournit le sucre (`sliding_window_view`, lui-même une vue) — écrivons la version nue et vérifions l'égalité :

Les pas de la sortie `(8, 8)` sont la signature du chevauchement : avancer d'une **fenêtre** (8 octets) et avancer d'un **élément dans la fenêtre** (8 octets aussi) coûtent le même déplacement — les fenêtres glissent d'un élément à la fois, donc chaque valeur du buffer sert dans plusieurs fenêtres simultanément. L'égalité vérifiée contre `sliding_window_view` dit que le sucre officiel et la construction nue `as_strided` fabriquent le même contrat d'adresse — le sucre n'ajoute que les garde-fous de bornes que la version nue n'a pas.

In [15]:
seq = np.arange(8, dtype=np.int64)
w = 3
fenetres = as_strided(seq, shape=(len(seq) - w + 1, w), strides=(8, 8))
print("fenetres =\n", fenetres)
print("strides :", fenetres.strides, "| memoire partagee :", np.shares_memory(fenetres, seq))

attendu = np.lib.stride_tricks.sliding_window_view(seq, w)
print("egalite avec sliding_window_view :", np.array_equal(fenetres, attendu))

fenetres =
 [[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]
 [4 5 6]
 [5 6 7]]
strides : (8, 8) | memoire partagee : True
egalite avec sliding_window_view : True


Six fenêtres de trois éléments : **18 valeurs logiques, 8 stockées**. Ce mécanisme est l'`im2col` de la convolution — déplier chaque patch 3×3 d'une image en colonnes pour n'avoir plus qu'un produit matriciel. Écrivons-le à la main et vérifions contre une double boucle :

La sortie 2D généralise la comptabilité : forme `(4, 4, 3, 3)` — seize positions (grille 4×4 des coins possibles sur une image 6×6 avec des patchs 3×3), chacune tenant un patch 3×3, soit 144 valeurs logiques lues dans 36 stockées. Le ratio mesuré (quatre lectures par octet stocké, ici) vient du recouvrement : les patchs voisins partagent leurs lignes entières. Et le premier patch imprimé — colonnes 0-2 des lignes 0-2, valeurs 0 à 14 espacées de 6 — montre la forme `(3, 3)` du patch telle que la vue la découpe. C'est ce dépliage que les bibliothèques de convolution matérialisent avant leur produit matriciel : le GEMM n'a plus ensuite que des colonnes régulières à consommer.

In [16]:
img = np.arange(36, dtype=np.int64).reshape(6, 6)  # image 6x6
p = 3                                                # patchs 3x3, pas 1
H, L = img.shape
forme = (H - p + 1, L - p + 1, p, p)                 # (4, 4, 3, 3)
pas = (img.strides[0], img.strides[1], img.strides[0], img.strides[1])
patchs = as_strided(img, shape=forme, strides=pas)
print("patchs :", patchs.shape, "soit", patchs.size, "valeurs logiques,",
      img.size, "stockees")
print("premier patch =\n", patchs[0, 0])

# Verification : la double boucle lit-elle les memes valeurs ?
ok = True
for i in range(H - p + 1):
    for j in range(L - p + 1):
        if not np.array_equal(patchs[i, j], img[i:i + p, j:j + p]):
            ok = False
print("les 16 patchs coincident avec img[i:i+3, j:j+3] :", ok)

patchs : (4, 4, 3, 3) soit 144 valeurs logiques, 36 stockees
premier patch =
 [[ 0  1  2]
 [ 6  7  8]
 [12 13 14]]
les 16 patchs coincident avec img[i:i+3, j:j+3] : True


Le dépliage 2D à pas 1 est mesuré ci-dessus : seize positions, 144 valeurs logiques, 36 stockées. Le pas `s` entre positions est la dernière variable de la recette — c'est lui qu'une coude convolutionnelle règle quand elle « saute » des positions, et la comptabilité du stockage change avec lui.

### Exercice 4 — des fenêtres 2D avec chevauchement choisi

**Contexte** : `im2col` à pas `s` (le *stride* de convolution) généralise la recette.
**Objectif** : écrire `patchs_2d(img, p, s)` qui extrait par `as_strided` tous les patchs `p×p` avec un décalage `s` entre patchs, vérifiable contre une double boucle.

```python
# Indice : le nombre de positions par axe est (H - p) // s + 1.
# Etape 1 : former shape = (n_pos_h, n_pos_l, p, p).
# Etape 2 : strides = (s * st0, s * st1, st0, st1) ou st = img.strides.
# Etape 3 : verifier contre img[i*s : i*s+p, j*s : j*s+p] pour tout (i, j).
```

In [17]:
def patchs_2d(img, p, s):
    '''Tous les patchs p x p de img, positions espacees de s, par as_strided.'''
    # TODO etudiant
    return None  # TODO etudiant


# Auto-verification (a decommenter une fois completee) :
# pv = patchs_2d(img, 3, 2)
# assert pv.shape == (2, 2, 3, 3)
# assert all(np.array_equal(pv[i, j], img[i*2:i*2+3, j*2:j*2+3]) for i in range(2) for j in range(2))
print("Exercice a completer")

Exercice a completer


C'est exactement la lecture que font les kernels de convolution (et, plus près de nous, les blocs d'attention par fenêtre glissante du [3.4](3.4-Attention-Transformer-From-Scratch.ipynb) et de la série [TransformerVariants](../../../GenAI/Texte/TransformerVariants/README.md)) : déplier sans copier, puis laisser le produit matriciel travailler.

L'exercice 4 ci-dessous ajoute le dernier degré de liberté — le pas `s` entre fenêtres, qui est le *stride* de convolution au sens des réseaux : la même recette, avec les pas de position multipliés par `s`. Complété, il donne la pièce maîtresse pour relire n'importe quelle couche convolutive comme un produit matriciel sur une vue.

## 6. Aliasing : le piège des mémoires partagées

Si deux tableaux partagent le buffer, **modifier l'un modifie l'autre**. Ce n'est pas un bug de NumPy — c'est la conséquence directe de la section 1, et un piège quotidien dès qu'on croit travailler sur une copie :

Le scénario est minimal à dessein : une « victime », une vue sur sa première ligne, une écriture `999` dans la vue. Le retour à la victime montre le `999` installé — la vue n'a jamais possédé de mémoire, elle a écrit *dans* le buffer de la victime, à l'offset que son contrat d'adresse désignait. `shares_memory : True` en est la preuve formelle. Le réflexe défensif tient en une question à se poser avant toute écriture : *ce tableau est-il une vue ?* — et, dans le doute, la copie explicite (`copy()` ou `np.ascontiguousarray`) coupe le lien.

In [18]:
victime = np.arange(6, dtype=np.int64).reshape(2, 3)
vue = victime[0]           # "la premiere ligne" -- une VUE
vue[1] = 999
print("victime apres modification de la vue =\n", victime)
print("shares_memory :", np.shares_memory(vue, victime))
print("-> la ligne 0 et la vue sont la meme memoire")

victime apres modification de la vue =
 [[  0 999   2]
 [  3   4   5]]
shares_memory : True
-> la ligne 0 et la vue sont la meme memoire


Et côté torch, la même mémoire traversée par l'**autograd** : le gradient suit les vues, parce qu'une vue est une opération (différentiable) sur le tenseur d'origine, pas un nouveau stockage :

Le gradient imprimé `[0., 2., 4., 6., 8., 10.]` se décode à la main depuis le code : la chaîne est `view(2,3)` puis `t()` puis `** 2` puis `sum()`, donc chaque élément de `x` n'a subi qu'une élévation au carré avant la somme — et la dérivée de $x^2$ est $2x$, qui pour $x = 0..5$ donne exactement les valeurs imprimées. La leçon tient au trajet, pas au chiffre : le gradient a été calculé **à travers** la vue et la transposée sans que jamais le stockage soit dupliqué — l'autograd traite les contrats d'adresse comme des opérations réversibles. C'est ce qui rend économiquement possible le graphique de calcul des frameworks : des millions de vues et de transposées traversées par la dérivation, sans une copie de plus.

In [19]:
x = torch.arange(6, dtype=torch.float64).requires_grad_()
y = x.view(2, 3).t() ** 2      # vue -> transposee -> carre : tout en gradients
y.sum().backward()
print("x.grad =", x.grad)
print("le gradient a traverse vue ET transposee, sans copie du stockage")

x.grad = tensor([ 0.,  2.,  4.,  6.,  8., 10.], dtype=torch.float64)
le gradient a traverse vue ET transposee, sans copie du stockage


## 7. Synthèse — ce qu'on emporte

| Geste | Ce qu'il fait vraiment | Coût mesuré dans ce carnet |
|---|---|---|
| `T`, slicing, `[::2, ::2]` | réécrit (offset, strides), même buffer | 0 octet — `shares_memory` vrai |
| `broadcast_to` / `expand` | stride **nul** sur l'axe diffusé | une ligne stockée pour N logiques |
| `repeat` | matérialise chaque copie | N lignes stockées |
| `.contiguous()` | copie bien rangée | le buffer au complet, une fois |
| `view` (torch) | exige la contiguïté | `RuntimeError` sinon — témoin §4 |
| fenêtres / `im2col` | strides qui se chevauchent | 18 logiques / 8 stockées (1D) · 144 / 36 (2D) |

**Trois idées à retenir.** Un tableau est un **contrat de lecture** (buffer, offset, forme, pas), pas une structure de données. Tout ce qui « ne copie pas » ne fait que réécrire ce contrat — gratuit en mémoire, parfois payé en **ordre d'accès** (le débit par élément mesuré en §3). Et le piège symétrique : deux tableaux, une mémoire — l'*aliasing* de la section 6, qui devient un atout quand l'autograd le traverse.

Le fil relierait presque chaque section à la suivante par une opposition : la gratuité (§2) a son prix caché (§3, l'ordre), le contrat unique a ses deux dialectes (§4, torch et NumPy), le chevauchement des fenêtres (§5) n'est que la mutualisation poussée de la même adresse, et l'aliasing (§6) est la face sombre de la même mutualisation. Une seule mécanique, quatre régimes d'emploi.

Le notebook suivant ouvre la mécanique qui exploite ces tenseurs pour apprendre : [3.1 — Rétropropagation](3.1-Retropropagation.ipynb).